In [ ]:
import csv
import string
import re
import urllib.request

def loading_reviews(clothing_reviews, limit=400):
  '''
  This function reads a CVS file containing clothing reviews, taking the csv file
  and the number of reviews to be classified as parameters.
  It then creates a list of dictionary containing the review and the reviews label
  (positive if the ratings 4 or above and negative if it is 2 or less.)
  The function returns the list of dictionaries.
  '''

  url = https://raw.github.com/jess-kellett/AI-coursework$0
  response = urllib.request.urlretrieve(url, "clothing_reviews.csv")
  reviews = []
  with open(clothing_reviews, 'r') as reviews_file:
    reader = csv.DictReader(reviews_file)
    for row in reader:
      if len(reviews) >= limit:
        break
      rating = int(row["Rating"])
      if rating >= 4:
        label = "positive"
      elif rating <= 2:
        label = "negative"
      # Skips rating 3 - ambiguous reviews could be positive or negaive so
      # including them would make evaluation unfair and reduce the classifiers reliablity
      else:
        continue
      review_text = row["Review Text"]
      if not review_text or not review_text.strip():
        continue
      reviews.append({
          "review" : row["Review Text"],
          "label" : label
          })
  return reviews

def preprocess_cleaning_reviews(reviews_list):
  """
  This function takes a list of dictionaries containing reviews and their labels
  as a parameter. The function then sorts through this list and cleans it.
  This includes making it all lowercase, removing punctuation and removing stop words.
  It returns the cleaned reviews as another list of dictionaries.
  """
  cleaned_reviews = []
  for reviews in reviews_list:
    # All reviews are converted into lowercase and all puntuation is removed to
    # ensure the same word is treated in the same way and matches the keywords.
    review_text = reviews["review"].lower().translate(str.maketrans('', '', string.punctuation))
    # stopwords are common words that add no sentiment value
    # they have been removed to help focus the classifier on meaningful words.
    stop_words = {"the", "a", "an", "is", "was", "and", "to", "of", "it", "this", "i", "has", "have", "its", "in", "im", "on", "am", "get", "my", "but", "as", "be", "so", "with"}
    words_in_reviews = review_text.split()
    words_in_reviews = [word for word in words_in_reviews if word not in stop_words]
    cleaned_reviews.append({"review": " ".join(words_in_reviews),
                           "label": reviews["label"]})
  return cleaned_reviews

def classifying_reviews(cleaned_review):
  """
  The parameter of this function is a string containing the cleaned review. This
  function classifies each review into either positive, negative or neutral
  depending on how many positive or negative keywords are present. It handles
  negation of both the positive and negative keywords to increase accuracy. It
  returns the classification outcome - "positive", "negative" or "neutral".
  """

  # Regex patterns are used to catch word variants.
  positive_keywords = ["wonderful", "sexy", r"comfortabl\w*", r"love\w*", "pretty", "glad", "fun", r"flirt\w*", r"fabulous\w*", "great", r"compliment\w*", r"flatter\w*", r"perfect\w*", r"gorgeous\w*", r"nice\w*", "feminine", r"beautiful\w*", "cute", "soft", "unique", "cozy", "better"]
  negative_keywords = [r"bad\w*", "unflattering", r"uncomfortable\w*", r"return\w*", "hate", "cheap", "awkward", r"disappoint\w*", r"itch\w*", "zero support", "seethrough"]
  negation_words = {"not", "never", "no", "doesnt", "wont", "dont"}
  override_negatives = [r"return\w*", r"refund\w*", "sending back"]

  review_words = cleaned_review.split()
  positive_count = 0
  negative_count = 0
  for index, words in enumerate(review_words):
    # Override checks occurs first as customers who have returned items have
    # overall disliked the product no matter how many positive attributes they have mentioned.
    for pattern in override_negatives:
      if re.search(pattern, words):
        return "negative"
    for pattern in positive_keywords:
      if re.search(pattern, words):
        # the word before the keyword is always checked to ensure it is not a
        # negation word, which would change the meaning and label completely.
        if index > 0 and review_words[index - 1] in negation_words:
          negative_count += 1
        else:
          positive_count += 1
    for pattern in negative_keywords:
      if re.search(pattern, words):
        if index > 0 and review_words[index - 1] in negation_words:
          positive_count += 1
        else:
          negative_count += 1
  if positive_count > negative_count:
    return "positive"
  elif positive_count < negative_count:
    return "negative"
  else:
    return "neutral"

def evaluate(cleaned_reviews):
  """
  This function recieves the list of dictionaries containing the cleaned reviews
  and their corresponding labels. It compares the labels of each review with the
  predicted classifations and returns the accuracy of the classifier, total reviews
  and total correctly classified reviews.
  """

  correctly_classified = 0
  total_reviews = 0

  for review in cleaned_reviews:
    review_text = review["review"]
    true_label = review["label"]

    predicted_by_algorithm = classifying_reviews(review_text)

    if predicted_by_algorithm == true_label:
      correctly_classified += 1

    if predicted_by_algorithm != true_label:
      print(f"Incorrect - Predicted: {predicted_by_algorithm}, Actual: {true_label}")
      print(f"Review: {review_text}\n")


    total_reviews += 1

  accuracy = correctly_classified / total_reviews
  return accuracy, total_reviews, correctly_classified


reviews = loading_reviews("clothing_reviews.csv")
cleaned_reviews = preprocess_cleaning_reviews(reviews)

accuracy, total_reviews, correctly_classified = evaluate(cleaned_reviews)

print(f"Accuracy: " + str(accuracy))
print(f"Total Reviews: " + str(total_reviews))
print(f"Correctly Classified: " + str(correctly_classified))




In [ ]:
# -------- Test section --------
# This section tests my classifier against 6 different edge cases to ensure the system behaves as expected.
# When running this section, please run my main code first.

print("---- Test cases ----")

test_cases = [
    {
        "title": "Test case 1: Clear positive review",
        "review": "I love this dress. It was super comfortable and the pattern was lovely. The design is so fun and pretty. I'm super glad I ordered it.",
        "expected_label": "positive"
    },
    {
        "title": "Test case 2: Clear negative review",
        "review": "I hate this dress. It was super uncomfortable and the pattern looks cheap and seethrough. The material is very itchy, I'm really disappointed.",
        "expected_label": "negative"
    },
    {
        "title": "Test case 3: Negation handling",
        "review": "This dress was not comfortable and not pretty",
        "expected_label": "negative"
    },
    {
        "title": "Test case 4: Override case - returned item",
        "review": "As much as I loved this dress and the pattern was fun, it didnt fit perfectly so I had to return it",
        "expected_label": "negative"
    },
    {
        "title": "Test case 5: Equally mixed review",
        "review": "I love this dress yet it is itchy",
        "expected_label": "neutral"
    },
    {
        "title": "Test case 6: No keywords",
        "review": "I ordered this dress for a special occasion",
        "expected_label": "neutral"
    }
  ]

passed = 0
for test in test_cases:
  cleaned_test_reviews = preprocess_cleaning_reviews([{"review": test["review"], "label": "positive"}])
  predicted_label = classifying_reviews(cleaned_test_reviews[0]["review"])
  result = "Passed" if predicted_label == test["expected_label"] else "Failed"
  if predicted_label == test["expected_label"]:
    passed += 1
  print(f"{test['title']} \nReview: {test['review']} \nExpected: {test['expected_label']} \nPredicted: {predicted_label} \nResult: {result} \n")

print(f"Passed: {passed}/{len(test_cases)}")
